In [ ]:
# Example 1: Public API surface is DYMGroup and DYMCommandCollection.
from click_didyoumean import DYMGroup, DYMCommandCollection

import click

print(f"DYMGroup: {DYMGroup}")
print(f"DYMCommandCollection: {DYMCommandCollection}")

# DYMGroup should subclass click.Group
assert issubclass(DYMGroup, click.Group), "DYMGroup must subclass click.Group"
# DYMCommandCollection should subclass click.CommandCollection
assert issubclass(DYMCommandCollection, click.CommandCollection), \
    "DYMCommandCollection must subclass click.CommandCollection"
print("Inheritance OK")

In [ ]:
# Example 2: Build a DYMGroup with subcommands and inspect it.
import click
from click_didyoumean import DYMGroup

@click.group(cls=DYMGroup)
def cli():
    """My CLI."""

@cli.command()
def install():
    """Install something."""
    click.echo("installing")

@cli.command()
def uninstall():
    """Uninstall something."""
    click.echo("uninstalling")

assert isinstance(cli, DYMGroup), "cli must be a DYMGroup instance"
assert set(cli.commands.keys()) == {"install", "uninstall"}
print(f"Commands registered: {sorted(cli.commands.keys())}")

In [ ]:
# Example 3: Invoke a valid command via CliRunner and check it works.
import click
from click.testing import CliRunner
from click_didyoumean import DYMGroup

@click.group(cls=DYMGroup)
def cli():
    pass

@cli.command()
def install():
    click.echo("installed")

runner = CliRunner()
result = runner.invoke(cli, ["install"])
print(f"exit_code={result.exit_code}")
print(f"output={result.output!r}")
assert result.exit_code == 0
assert "installed" in result.output

In [ ]:
# Example 4: Mistyped command should produce a 'did you mean' suggestion.
import click
from click.testing import CliRunner
from click_didyoumean import DYMGroup

@click.group(cls=DYMGroup)
def cli():
    pass

@cli.command()
def install():
    click.echo("installed")

@cli.command()
def uninstall():
    click.echo("uninstalled")

runner = CliRunner()
# 'instal' is a typo for 'install' - should trigger suggestion
result = runner.invoke(cli, ["instal"])
print(f"exit_code={result.exit_code}")
print(f"output={result.output!r}")
# Non-zero exit (command not found), and 'install' should appear in suggestion text
assert result.exit_code != 0
assert "install" in result.output.lower()

In [ ]:
# Example 5: DYMGroup accepts max_suggestions and cutoff parameters.
import click
from click.testing import CliRunner
from click_didyoumean import DYMGroup

@click.group(
    cls=DYMGroup,
    max_suggestions=2,
    cutoff=0.5,
)
def cli():
    pass

@cli.command()
def push():
    click.echo("push")

@cli.command()
def pull():
    click.echo("pull")

@cli.command()
def status():
    click.echo("status")

# Confirm tuning attributes were stored
assert cli.max_suggestions == 2
assert cli.cutoff == 0.5

runner = CliRunner()
result = runner.invoke(cli, ["pus"])  # close to 'push' and 'pull'
print(f"exit_code={result.exit_code}")
print(f"output={result.output!r}")
assert result.exit_code != 0
print("max_suggestions/cutoff configurable OK")